# Week 9 Lab — Markov Switching and State-Space Models

**MANG2074 Financial Econometrics 1**

**Objectives**

- Estimate a two-regime Markov switching model with switching mean and variance.
- Interpret transition probabilities, expected durations and smoothed probabilities.
- Date bull/bear regimes against market history.
- Meet the Kalman filter via a local-level state-space model.

**Data**

`../data/data_monthly.csv` — monthly returns (decimal), 1985–2023.


## Setup

Load the monthly return series (we scale to % per month).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

d = pd.read_csv('../data/data_monthly.csv', index_col=0, parse_dates=True).asfreq('MS')
r = (100 * d['return']).dropna()        # convert to % per month
r.plot(figsize=(10, 4), lw=0.8, title='Monthly returns (%), 1985-2023')
plt.show()
print(r.describe().round(2))

## Task 1 — Why a single-regime model fails

Compute the mean and standard deviation of returns over the full sample, and then separately for 2003–2007 and 2008–2009. Do these look like draws from one distribution?

In [ ]:
# YOUR CODE HERE: full-sample, 2003-2007 and 2008-2009 mean/std


## Task 2 — Estimate the Markov switching model

Fit a two-regime model with switching mean **and** switching variance:

```python
mod = sm.tsa.MarkovRegression(r, k_regimes=2, switching_variance=True)
res = mod.fit()
```

Print the summary. Identify which regime is 'calm/bull' and which is 'turbulent/bear' from the estimated means and variances (convert $\sigma^2$ to monthly vol).

In [ ]:
mod = sm.tsa.MarkovRegression(r, k_regimes=2, switching_variance=True)
res = mod.fit()
# YOUR CODE HERE: print summary; compute the regime vols


## Task 3 — Transition probabilities and durations

From the summary, read $p_{00}$ and $p_{11}$ (note statsmodels reports `p[0->0]` and `p[1->0]`). Compute the expected duration of each regime by hand, $1/(1-p_{ii})$, and check against `res.expected_durations`. Interpret in months/years.

In [ ]:
# YOUR CODE HERE


## Task 4 — Smoothed regime probabilities

Plot `res.smoothed_marginal_probabilities[1]` (probability of the high-variance regime) as a filled area, with the return series in a panel above. Which historical episodes does the model assign to the turbulent regime?

In [ ]:
# YOUR CODE HERE: two-panel plot (returns; smoothed probability of turbulent regime)


## Task 5 — Regime dating

Create a dummy `bear = (smoothed prob of turbulent regime > 0.5)`. List the start and end dates of each turbulent spell longer than 3 months, and compute the mean return within bear vs bull months.

In [ ]:
prob_bear = res.smoothed_marginal_probabilities[1]
bear = prob_bear > 0.5
# YOUR CODE HERE: find spells (hint: bear.ne(bear.shift()).cumsum() groups consecutive runs)


## Task 6 — Markov autoregression

Fit `sm.tsa.MarkovAutoregression(r, k_regimes=2, order=1, switching_variance=True)`. Does adding within-regime AR(1) dynamics change the regime story? Compare log-likelihoods (note: the MS-AR uses one fewer observation).

In [ ]:
# YOUR CODE HERE


## Task 7 — Taster: the Kalman filter and a local-level model

Estimate the local-level model $r_t = \mu_t + \epsilon_t$, $\mu_t = \mu_{t-1} + \eta_t$ with `sm.tsa.UnobservedComponents(r, 'local level')`. Plot the smoothed level state over the returns. What is this model saying the 'underlying mean return' did over time?

In [ ]:
uc = sm.tsa.UnobservedComponents(r, 'local level')
uc_res = uc.fit(disp=False)
# YOUR CODE HERE: plot r and uc_res.smoothed_state[0]


## Task 8 — Interpretation

5–7 sentences: describe the two regimes (mean, vol, duration); why are smoothed probabilities unusable as a real-time trading signal; one finance question you would attack with a state-space model.

*YOUR ANSWER HERE*